# Modelo de Machine Learning — Titanic

## Objetivo

Nesta etapa, será construído um modelo de classificação para prever se um passageiro sobreviveu ao naufrágio do Titanic.

O arquivo `train.csv` será utilizado para treinar e validar o modelo, pois contém a variável-alvo `Survived`. Depois da avaliação, o modelo será treinado novamente com todos os registros disponíveis e utilizado para gerar previsões para o arquivo `test.csv`.


## Etapas do notebook

1. Carregamento das bases de treino e teste;
2. Criação de novas variáveis;
3. Seleção das variáveis explicativas e da variável-alvo;
4. Separação entre treino e validação;
5. Tratamento de valores ausentes e variáveis categóricas;
6. Treinamento de uma Regressão Logística;
7. Avaliação do modelo;
8. Validação cruzada;
9. Treinamento com todos os dados;
10. Geração do arquivo `submission.csv`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import (
    cross_val_score,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

## 1. Carregamento dos dados

O conjunto de treino contém a coluna `Survived`, enquanto o conjunto de teste não possui essa coluna. Portanto, não é possível medir diretamente o desempenho do modelo no `test.csv`.


In [ ]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(f"Treino: {train.shape[0]} linhas e {train.shape[1]} colunas")
print(f"Teste: {test.shape[0]} linhas e {test.shape[1]} colunas")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
print("Colunas presentes apenas no treino:")
print(sorted(set(train.columns) - set(test.columns)))

### Interpretação

A coluna `Survived` aparece apenas no conjunto de treino porque ela representa a resposta que o modelo deverá aprender a prever.


## 2. Engenharia de atributos

Serão criadas duas variáveis simples:

- `FamilySize`: quantidade total de pessoas da família a bordo, incluindo o próprio passageiro;
- `IsAlone`: indica se o passageiro viajava sozinho.


In [ ]:
def criar_atributos(dados):
    dados = dados.copy()

    dados["FamilySize"] = (
        dados["SibSp"]
        + dados["Parch"]
        + 1
    )

    dados["IsAlone"] = (
        dados["FamilySize"] == 1
    ).astype(int)

    return dados


train_ml = criar_atributos(train)
test_ml = criar_atributos(test)

In [ ]:
train_ml[
    ["SibSp", "Parch", "FamilySize", "IsAlone"]
].head()

## 3. Seleção das variáveis

Para o primeiro modelo, serão utilizadas variáveis que apresentaram relação com a sobrevivência durante a análise exploratória.

As colunas `PassengerId`, `Name`, `Ticket` e `Cabin` não serão utilizadas neste modelo inicial.


In [ ]:
variaveis_numericas = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

variaveis_categoricas = [
    "Pclass",
    "Sex",
    "Embarked"
]

variaveis_modelo = (
    variaveis_numericas
    + variaveis_categoricas
)

X = train_ml[variaveis_modelo]
y = train_ml["Survived"]

X_test_kaggle = test_ml[variaveis_modelo]

In [ ]:
print("Variáveis explicativas:")
print(variaveis_modelo)

print("\nDistribuição da variável-alvo:")
print(y.value_counts(normalize=True).mul(100).round(2))

## 4. Separação entre treino e validação

Uma parte do `train.csv` será reservada para validação local. Dessa forma, o modelo será avaliado em passageiros que não foram utilizados durante o treinamento.

O parâmetro `stratify=y` mantém proporções semelhantes de sobreviventes e não sobreviventes nos dois subconjuntos.


In [ ]:
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Registros de treino: {len(X_treino)}")
print(f"Registros de validação: {len(X_validacao)}")

## 5. Pré-processamento

As variáveis numéricas e categóricas exigem tratamentos diferentes:

- valores ausentes numéricos serão preenchidos com a mediana;
- variáveis numéricas serão padronizadas;
- valores ausentes categóricos serão preenchidos com o valor mais frequente;
- categorias textuais serão transformadas em colunas numéricas por meio de One-Hot Encoding.

Todo o pré-processamento será reunido em um `Pipeline`, garantindo que os mesmos tratamentos sejam aplicados ao treino, à validação e ao conjunto de teste.


In [ ]:
pipeline_numerico = Pipeline(
    steps=[
        (
            "preenchimento",
            SimpleImputer(strategy="median")
        ),
        (
            "padronizacao",
            StandardScaler()
        )
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        (
            "preenchimento",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "codificacao",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            pipeline_numerico,
            variaveis_numericas
        ),
        (
            "categoricas",
            pipeline_categorico,
            variaveis_categoricas
        )
    ]
)

## 6. Criação e treinamento do modelo

A Regressão Logística será utilizada como modelo inicial. Apesar do nome, ela é um algoritmo de classificação adequado para prever resultados binários, como `0` ou `1`.


In [ ]:
modelo = Pipeline(
    steps=[
        (
            "preprocessamento",
            preprocessador
        ),
        (
            "classificador",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

modelo.fit(
    X_treino,
    y_treino
)

## 7. Avaliação na base de validação


In [ ]:
previsoes_validacao = modelo.predict(
    X_validacao
)

acuracia = accuracy_score(
    y_validacao,
    previsoes_validacao
)

print(f"Acurácia na validação: {acuracia:.2%}")

In [ ]:
print(
    classification_report(
        y_validacao,
        previsoes_validacao,
        target_names=[
            "Não sobreviveu",
            "Sobreviveu"
        ]
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_validacao,
    previsoes_validacao,
    display_labels=[
        "Não sobreviveu",
        "Sobreviveu"
    ],
    cmap="Blues"
)

plt.title("Matriz de confusão — validação")
plt.tight_layout()
plt.show()

### Como interpretar a avaliação

- **Acurácia:** proporção total de previsões corretas;
- **Precisão:** entre os passageiros previstos em uma classe, quantos realmente pertenciam a ela;
- **Recall:** entre os passageiros que realmente pertenciam a uma classe, quantos foram identificados;
- **F1-score:** equilíbrio entre precisão e recall;
- **Matriz de confusão:** mostra acertos e erros de cada classe.

A acurácia não deve ser analisada isoladamente. É importante observar se o modelo consegue identificar tanto sobreviventes quanto não sobreviventes.


## 8. Validação cruzada

A validação cruzada repete o treinamento e a avaliação em diferentes divisões dos dados. Isso fornece uma estimativa mais estável do desempenho do modelo do que uma única divisão.


In [ ]:
resultados_validacao_cruzada = cross_val_score(
    modelo,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Acurácias em cada divisão:")
print(resultados_validacao_cruzada.round(4))

print(
    "\nAcurácia média: "
    f"{resultados_validacao_cruzada.mean():.2%}"
)

print(
    "Desvio padrão: "
    f"{resultados_validacao_cruzada.std():.2%}"
)

## 9. Treinamento final

Após a avaliação, o modelo será treinado novamente utilizando todos os registros do `train.csv`.


In [ ]:
modelo.fit(
    X,
    y
)

## 10. Previsões para o conjunto de teste


In [ ]:
previsoes_teste = modelo.predict(
    X_test_kaggle
)

previsoes_teste[:10]

## 11. Criação do arquivo de submissão

O arquivo final deve possuir exatamente duas colunas:

- `PassengerId`;
- `Survived`.


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": previsoes_teste.astype(int)
})

submission.head()

In [ ]:
assert len(submission) == len(test)
assert list(submission.columns) == [
    "PassengerId",
    "Survived"
]
assert submission["Survived"].isin([0, 1]).all()
assert submission["PassengerId"].equals(
    test["PassengerId"]
)

print("Formato da submissão validado com sucesso.")

In [ ]:
caminho_submission = "../submission.csv"

submission.to_csv(
    caminho_submission,
    index=False
)

print(
    "Arquivo criado em:",
    caminho_submission
)

## 12. Conclusão

Foi construído um pipeline completo de classificação contendo:

- criação de atributos;
- tratamento de valores ausentes;
- transformação de variáveis categóricas;
- padronização de variáveis numéricas;
- treinamento de uma Regressão Logística;
- avaliação em uma base de validação;
- validação cruzada;
- geração de previsões para o `test.csv`;
- criação do arquivo `submission.csv`.

Como próximos passos, o desempenho poderá ser comparado com outros algoritmos, como Árvore de Decisão e Random Forest, além da criação de novos atributos a partir das colunas `Name`, `Ticket` e `Cabin`.
